# Seismic Parameter Computation
Compute mass, radius, log g from numax, dnu, Teff using asteroseismic scaling relations.
Applies APOKASC-3 calibration corrections and Sharma+2016 f_dnu model corrections.
Propagates uncertainties via first-order Taylor expansion (quadrature).

In [ ]:
import pandas as pd
import numpy as np

# Solar reference values (APOKASC-3)
NUMAX_SUN = 3090.0   # uHz
DNU_SUN = 135.1      # uHz
TEFF_SUN = 5772.0    # K
LOGG_SUN = 4.438     # dex

# APOKASC-3 calibration factors
X_NUMAX = 1 / 1.0009
X_DNU = 1 / 0.9973

# Measurement uncertainties
SIGMA_NUMAX_FRAC = 0.033  # 3.3%
SIGMA_DNU_FRAC = 0.03     # 3%
SIGMA_TEFF = 100           # K

In [ ]:
def estimate_fdnu(teff, dnu, feh):
    """Estimate f_dnu from Sharma+2016 approximation for RGB stars."""
    f = 0.978 + 0.006 * (teff - 4500) / 500 + 0.004 * (dnu - 5) / 5 + 0.002 * feh
    return np.clip(f, 0.970, 0.995)

def compute_seismic(numax_raw, dnu_raw, teff, feh):
    """Compute corrected seismic parameters and uncertainties."""
    # Corrections
    f_dnu = estimate_fdnu(teff, dnu_raw, feh)
    numax_corr = numax_raw * X_NUMAX
    dnu_corr = dnu_raw * X_DNU / f_dnu
    
    # Scaling relations
    R = (numax_corr / NUMAX_SUN) * (dnu_corr / DNU_SUN)**(-2) * (teff / TEFF_SUN)**0.5
    M = (numax_corr / NUMAX_SUN)**3 * (dnu_corr / DNU_SUN)**(-4) * (teff / TEFF_SUN)**1.5
    logg = np.log10((numax_corr / NUMAX_SUN) * (teff / TEFF_SUN)**0.5) + LOGG_SUN
    
    # Error propagation (quadrature)
    s_t = SIGMA_TEFF / teff
    R_err = R * np.sqrt(SIGMA_NUMAX_FRAC**2 + 4*SIGMA_DNU_FRAC**2 + 0.25*s_t**2)
    M_err = M * np.sqrt(9*SIGMA_NUMAX_FRAC**2 + 16*SIGMA_DNU_FRAC**2 + 2.25*s_t**2)
    logg_err = np.sqrt(SIGMA_NUMAX_FRAC**2 + 0.25*s_t**2) / np.log(10)
    
    return {
        'f_dnu': f_dnu, 'numax_corr': numax_corr, 'dnu_corr': dnu_corr,
        'R_seis': R, 'M_seis': M, 'logg_seis': logg,
        'R_err': R_err, 'M_err': M_err, 'logg_err': logg_err
    }

In [ ]:
# Load data and compute
df = pd.read_csv('../data/seismic_results.csv')

for idx, row in df.iterrows():
    if np.isnan(row.get('dnu_measured', np.nan)):
        continue
    result = compute_seismic(row['numax_final'], row['dnu_measured'], row['teff'], row['feh'])
    for k, v in result.items():
        df.loc[idx, k] = v

print(df[['TICID', 'cluster', 'numax_final', 'dnu_measured', 'R_seis', 'M_seis', 'logg_seis']].to_string())
df.to_csv('../data/seismic_results.csv', index=False)
print('Saved.')